# ACTIVE-TB-S: Physically Isolated Spec-Only Oracle Pilot

This phase sees only `task_id`, natural-language `description`, and the required module `header`. It generates two executable oracle candidates with Qwen and three with Llama 3.1. Golden RTL, official testbenches, and mutants are not present in the attached dataset.


In [ ]:
# 1) Locate the sanitized artifact and copy executable code to writable storage.
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/kaggle/working/hf_cache'

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
markers = list(INPUT.rglob('active_tb_s_oracle_pilot_v1/LEAKAGE_BOUNDARY.json'))
if len(markers) == 1:
    SOURCE = markers[0].parent
else:
    archives = list(INPUT.rglob('active_tb_s_oracle_pilot_v1.zip'))
    assert len(archives) == 1, {'markers': markers, 'archives': archives}
    extracted = WORK / 'attached_oracle_source'
    with zipfile.ZipFile(archives[0]) as zf: zf.extractall(extracted)
    SOURCE = extracted / 'active_tb_s_oracle_pilot_v1'
PROJECT = WORK / 'active_tb_s_project'
if PROJECT.exists(): shutil.rmtree(PROJECT)
shutil.copytree(SOURCE/'project', PROJECT)
SPEC_TASKS = SOURCE / 'spec_tasks_pilot.jsonl'
OUT = WORK / 'active_tb_s_oracle_candidates_v1'
OUT.mkdir(exist_ok=True)
print('Sanitized source:', SOURCE)


In [ ]:
# 2) Enforce the leakage boundary before any model is loaded.
boundary = json.loads((SOURCE/'LEAKAGE_BOUNDARY.json').read_text())
spec_rows = [json.loads(line) for line in SPEC_TASKS.read_text().splitlines() if line.strip()]
assert len(spec_rows) == 20
assert all(set(row) == {'task_id','description','header'} for row in spec_rows)
for forbidden in boundary['forbidden_fields']:
    assert all(forbidden not in row for row in spec_rows)
assert not list(SOURCE.rglob('*mutant*'))
print('SPEC_ONLY_LEAKAGE_AUDIT_PASS:', len(spec_rows), 'tasks')


In [ ]:
# 3) Install a GPU-compatible generation/runtime stack.
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','iverilog'], check=True)
# Kaggle may allocate a Pascal P100 even when a generic GPU is requested. Its
# newest base-image PyTorch can omit sm_60 kernels, so use a wheel that supports
# both P100 and T4 instead of relying on the mutable image default.
subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir','--force-reinstall',
                'torch==2.5.1','--index-url','https://download.pytorch.org/whl/cu121'], check=True)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','-q','torchvision','torchaudio'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir',
                'transformers==4.57.6','accelerate>=1.0,<2','bitsandbytes==0.48.2','tqdm>=4.66'], check=True)
gpu_probe = subprocess.run([sys.executable,'-c',
    'import torch; print(torch.__version__, torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0)); '
    'x=torch.ones(1,device="cuda"); print("CUDA_PROBE_PASS",x.item())'],
    text=True, capture_output=True)
print(gpu_probe.stdout)
if gpu_probe.returncode != 0: raise RuntimeError(gpu_probe.stderr)
subprocess.run([sys.executable, str(PROJECT/'spec_only_smoke_test.py')], check=True, cwd=PROJECT)


In [ ]:
# 4) Qwen-family independent candidates. Prequantized checkpoint keeps disk use bounded.
QWEN_MODEL = 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit'
QWEN_OUT = OUT / 'qwen_candidates.jsonl'
cmd = [sys.executable, str(PROJECT/'generate_spec_oracles.py'),
       '--tasks', str(SPEC_TASKS), '--output', str(QWEN_OUT),
       '--model', QWEN_MODEL, '--family', 'qwen', '--candidates-per-task', '2',
       '--max-new-tokens', '900', '--temperature', '0.20', '--repair-attempts', '1', '--resume']
subprocess.run(cmd, check=True, cwd=PROJECT)
print('Qwen rows:', len(QWEN_OUT.read_text().splitlines()))


In [ ]:
# 5) Release Qwen weights/cache before loading the independent Llama family.
import gc
gc.collect()
try:
    import torch
    torch.cuda.empty_cache()
except Exception: pass
cache = Path(os.environ['HF_HOME'])
if cache.exists(): shutil.rmtree(cache)
cache.mkdir(parents=True, exist_ok=True)
print('Qwen cache released')


In [ ]:
# 6) Meta Llama-family independent candidates.
LLAMA_MODEL = 'unsloth/Llama-3.1-8B-Instruct-unsloth-bnb-4bit'
LLAMA_OUT = OUT / 'llama_candidates.jsonl'
cmd = [sys.executable, str(PROJECT/'generate_spec_oracles.py'),
       '--tasks', str(SPEC_TASKS), '--output', str(LLAMA_OUT),
       '--model', LLAMA_MODEL, '--family', 'llama', '--candidates-per-task', '3',
       '--max-new-tokens', '900', '--temperature', '0.20', '--repair-attempts', '1', '--resume']
subprocess.run(cmd, check=True, cwd=PROJECT)
print('Llama rows:', len(LLAMA_OUT.read_text().splitlines()))


In [ ]:
# 7) Validate and package the checkpoint for the gold-free synthesis phase.
expected_rows = {QWEN_OUT: 40, LLAMA_OUT: 60}
for path in [QWEN_OUT, LLAMA_OUT]:
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    assert len(rows) == expected_rows[path], (path, len(rows))
    print(path.name, 'compiled', sum(bool(row['compiled']) for row in rows), '/', len(rows))
manifest = {
    'protocol': 'spec-only oracle generation',
    'task_count': 20,
    'input_fields': ['task_id','description','header'],
    'models': [QWEN_MODEL, LLAMA_MODEL],
}
(OUT/'oracle_manifest.json').write_text(json.dumps(manifest, indent=2))
archive = WORK / 'active_tb_s_oracle_candidates_v1.zip'
shutil.make_archive(str(archive.with_suffix('')), 'zip', OUT)
if cache.exists(): shutil.rmtree(cache)
print('Checkpoint:', archive, archive.stat().st_size, 'bytes')
